# 🌳 두번째 함수 만들기

**날짜**: 2026-05-18  
**목표**: 함지산 산불 전후 평균 NDVI 추출하기

## 🛠️ 만들 함수
1. `get_mean_ndvi()`: NDVI 영상에서 평균값 1개 추출

In [ ]:
# 라이브러리 가져오기
import ee
import geemap

# Earth Engine 초기화
ee.Initialize(project='knu-project-ndvi-2026')

In [ ]:
# 1st_create_functions에서 만든 함수 재사용

def get_sentinel2_image(area, start_date, end_date, cloud_threshold=10):
    """Sentinel-2 영상 1장 가져오기"""
    image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
             .filterBounds(area)
             .filterDate(start_date, end_date)
             .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
             .first())
    return image


def calculate_ndvi(image):
    """NDVI 계산"""
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ndvi


print("✅ 함수 로딩 완료!")

## 🗺️ 분석 영역 정의

함지산 좌표 1개로는 평균 못 냄.  
**buffer()로 점을 원형 영역으로 확장** 필요.

- 중심점: [128.6014, 35.9156] (함지산)
- 반경: 2500m (2.5km)

## 📌 버퍼(Buffer)란?

버퍼(Buffer)는 GIS에서 가장 많이 활용되는 **공간 분석 도구** 중 하나로, 특정 지점을 중심으로 **정해진 거리 만큼의 영향권(범위)** 을 시각화할 때 사용됨.

In [ ]:
# 분석 영역: 함지산 중심 + 반경 800m 원형 영역

center_point = ee.Geometry.Point([128.5774, 35.9145])
study_area = center_point.buffer(800)   # 800m 
                                        # 점 1개 -> 반경 800m 원형 영역으로 확장

print("✅ 분석 영역 정의 완료")
print("중심: 함지산 [128.5774, 35.9145]")
print("반경: 800m")

## 🛠️ 함수: get_mean_ndvi

NDVI 영상에서 분석 영역의 평균값을 추출.

### 📥 입력
- `ndvi_image`: NDVI 영상
- `area`: 분석 영역 (buffer로 만든 면적)

### 📤 출력
- 평균 NDVI 값 (예: 0.65)

In [ ]:
def get_mean_ndvi(ndvi_image, area):
    """
    NDVI 영상에서 평균값을 추출한다.
    
    매개변수:
        ndvi_image (ee.Image): NDVI 영상
        area (ee.Geometry): 분석 영역
    
    반환값:
        float: 평균 NDVI 값
    """
    # 영역 내 평균 계산
    
    mean_dict = ndvi_image.reduceRegion(
        reducer = ee.Reducer.mean(),    # 평균 계산기
        geometry = area,                # 영역
        scale = 10,                     # 픽셀 크기 (Sentinel-2 = 10m)
        maxPixels = 1e9                 # 최대 픽셀 (안전장치)
    )
    
    # 실제 숫자 추출
    
    mean_value = mean_dict.get('NDVI').getInfo()
    
    return mean_value


print("✅ get_mean_ndvi() 함수 정의 완료!")

In [ ]:
# 산불 전 (2025년 4월 초)

image_before = get_sentinel2_image(study_area, '2025-04-01', '2025-04-20')
ndvi_before = calculate_ndvi(image_before)
mean_before = get_mean_ndvi(ndvi_before, study_area)

print(f"산불 전 평균 NDVI: {mean_before:.3f}")

In [ ]:
# 산불 후 (2025년 5월)

image_after = get_sentinel2_image(study_area, '2025-04-29', '2025-05-15')
ndvi_after = calculate_ndvi(image_after)
mean_after = get_mean_ndvi(ndvi_after, study_area)

print(f"산불 후 평균 NDVI: {mean_after:.3f}")

In [ ]:
# 변화량 계산

diff = mean_after - mean_before
percent_change = (diff / mean_before) * 100

# 결과 출력

print("=" * 40)
print("🔥 함지산 산불 NDVI 분석 결과")
print("=" * 40)
print(f"산불 전 평균 NDVI: {mean_before:.3f}")
print(f"산불 후 평균 NDVI: {mean_after:.3f}")
print(f"변화량: {diff:.3f}")
print(f"변화율: {percent_change:.1f}%")
print("=" * 40)

In [ ]:
# 시각화 설정

ndvi_vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['white', 'yellow', 'lightgreen', 'green', 'darkgreen']
}

# 비교 지도

Map = geemap.Map(center=[35.9145, 128.5774], zoom=13)
Map.addLayer(ndvi_before, ndvi_vis_params, f'산불 전 (평균 {mean_before:.3f})')
Map.addLayer(ndvi_after, ndvi_vis_params, f'산불 후 (평균 {mean_after:.3f})')

# 분석 영역 표시 (빨간 원)

Map.addLayer(study_area, {'color': 'red'}, '분석 영역 (800m)')

Map

In [ ]:
print("산불 전 영상 날짜:", ee.Date(image_before.get('system:time_start')).format('YYYY-MM-dd').getInfo())
print("산불 후 영상 날짜:", ee.Date(image_after.get('system:time_start')).format('YYYY-MM-dd').getInfo())

In [ ]:
# 같은 해의 4월과 5월달을 비교하니 4월에는 식생이 무성하게 자라지 않아 비교 불가능 판단(차이가 적음)
# 같은 계절로 그 직전 년도인 2024년과 2025년을 비교 진행

# 산불 전 = 작년 5월 (정상 식생)
image_before = get_sentinel2_image(study_area, '2024-05-01', '2024-05-31')
ndvi_before = calculate_ndvi(image_before)
mean_before = get_mean_ndvi(ndvi_before, study_area)

# 산불 후 = 올해 5월 (산불 피해)
image_after = get_sentinel2_image(study_area, '2025-05-01', '2025-05-31')
ndvi_after = calculate_ndvi(image_after)
mean_after = get_mean_ndvi(ndvi_after, study_area)

# 날짜 + 결과
print("전(작년):", ee.Date(image_before.get('system:time_start')).format('YYYY-MM-dd').getInfo())
print("후(올해):", ee.Date(image_after.get('system:time_start')).format('YYYY-MM-dd').getInfo())
print(f"산불 전 NDVI: {mean_before:.3f}")
print(f"산불 후 NDVI: {mean_after:.3f}")
print(f"변화율: {((mean_after - mean_before) / mean_before * 100):.1f}%")

In [ ]:
# 시각화 설정

ndvi_vis_params = {
    'min': 0,
    'max': 1,
    'palette': ['white', 'yellow', 'lightgreen', 'green', 'darkgreen']
}

# 비교 지도

Map = geemap.Map(center=[35.9145, 128.5774], zoom=13)
Map.addLayer(ndvi_before, ndvi_vis_params, f'산불 전 (평균 {mean_before:.3f})')
Map.addLayer(ndvi_after, ndvi_vis_params, f'산불 후 (평균 {mean_after:.3f})')

# 분석 영역 표시 (빨간 원)

Map.addLayer(study_area, {'color': 'red'}, '분석 영역 (800m)')

Map